# RAG Agent -- Process Check

- Create a LLM client
- RAG Prep: register OpenAIEmbeddings, pdf_path, pdf_loader, text_splitter 
- Register page split and the rest of RAG configurations including persist_directory, collection_name
- Make a vectorstore and retriever
- Make a retriever_tool and configure the tool settings
- Set langchain agent state
- Make conditional edge function
- Create the main llm agent fucntion
- Make a retriever agent that acts as manual ToolNode
- Register graph order
- Create the execution function

In [32]:
from dotenv import load_dotenv
import os
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools import tool

In [14]:
load_dotenv()

True

In [15]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

In [16]:
pdf_path = "Stock_Market_Performance_2024.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF file not found: {pdf_path}")

pdf_loader = PyPDFLoader(file_path=pdf_path) 

In [17]:
try:
    pages = pdf_loader.load()
    print(f"PDF has been loaded and has {len(pages)} pages")
except Exception as e:
    print(f"Error loading PDF: {e}")
    raise

PDF has been loaded and has 9 pages


In [27]:
# Chunking Process
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

pages_split = text_splitter.split_documents(pages)

persist_directory =r"/Volumes/VTG/Dev/LangGraph tutorial"
collection_name = "stock_market"

In [28]:
if not os.path.exists(persist_directory): 
    os.makedirs(persist_directory)

In [33]:
try: 
    # Create Chroma Database 
    vectorstore = Chroma.from_documents(
        documents=pages_split,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name
    )
    print("Created ChromaDB vector store!")
except Exception as e: 
    print(f"Error setting up ChromaDB: {str(e)}")
    raise

Created ChromaDB vector store!


In [34]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5} # K is the amount of chunks to return
)